# islide — M4 annotation editing demo

M4 is **editing**: a shape that already exists — imported (M2) or drawn
(M3) — can be selected in the view and deleted, labeled, or recolored.
There is no geometry editing (no move / resize / vertex edit) and no
multi-select; `v.annotations` stays the single source of truth, and the
view's buttons issue the same commands the Python API below issues.

## 1. Open the slide and import a document

The slide opens on a background thread; `wait()` blocks until it's
ready. The imported document carries explicit ids (`roi`, `nucleus`),
which the editing API below addresses.

In [ ]:
from islide import SlideViewer

# Test slide (data/ is gitignored): download data/CMU-1.tiff from
# https://openslide.cs.cmu.edu/download/openslide-testdata/Generic-TIFF/CMU-1.tiff
v = SlideViewer("data/CMU-1.tiff")
v.wait()

DOC = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "id": "roi",
            "geometry": {
                "type": "Polygon",
                "coordinates": [[[22700, 16150], [23300, 16150],
                                  [23300, 16750], [22700, 16750]]],
            },
            "properties": {},
        },
        {
            "type": "Feature",
            "id": "nucleus",
            "geometry": {"type": "Point", "coordinates": [23000, 16450]},
            "properties": {"label": "nucleus"},
        },
    ],
}
v.set_annotations(DOC)
v.center_on(23000, 16450)
v.set_zoom(8.0)
v

## 2. Select and edit in the view

In the idle mode (not drawing):

1. **Left-click a shape** to select it — it is highlighted with a thick
   accent stroke (a point gets an accent ring). Clicking empty space
   deselects; clicking another shape moves the selection. A drag of ≥ 4
   px is still a pan, and navigation (pan / zoom / minimap) keeps the
   selection on the shape.
2. While a shape is selected, three toolbar buttons are enabled:
   - **label** — an inline text input appears in the toolbar, prefilled
     with the current label. Enter (or blur) commits; an empty string
     clears the label; Esc cancels. M4 also renders labels on
     lines/polygons (at the first vertex), not just points.
   - **color** — two native color pickers (stroke, fill) plus a
     clear-fill checkbox. Closing either picker commits the full
     `(color, fill)` pair, so you can set the stroke only, the fill
     only, both, or clear the fill.
   - **del** — removes the selected shape.
3. The **annotate** toolbar button toggles the M3 drawing mode (**A**
   stays the keyboard alias); entering drawing mode clears the selection.

Every edit lands back in `v.annotations` (the whole set is pushed), and
the status bar reports it (`deleted #<id>` / `edited #<id> (label|color)`).
After **label** / **color** the selection persists, so edits chain.

## 3. The same ops programmatically

The view's del / label / color buttons issue `annotation_edit` commands
over the comm channel; the public API issues the same commands through
the same trait (ids come from `v.annotations`):

```python
v.delete_annotation(feature_id)
v.set_annotation_label(feature_id, "tumor")   # None / "" clears
v.set_annotation_color(feature_id, color="red", fill=None)  # None = default
```

Unknown or stale ids (e.g. after a `set_annotations()` replace) are
no-ops: the document is untouched and the status says `edit ignored:`.
An id that no longer exists because the shape was deleted in the view is
one of these — the command is a harmless replay.

In [ ]:
v.set_annotation_label("nucleus", "tumor")
v.set_annotation_color("roi", color="red", fill="rgba(255,0,0,0.2)")
v.annotations

## 4. Draw one (M3) and delete it

Press **A** (or the **annotate** button), left-click the vertices of a
new polygon, press **A** to save. Select it and press **del** in the
view — or do it here (hand-drawn features get fresh `aN` ids, so they
can be told apart from the imported `roi` / `nucleus`):


In [ ]:
# remove the hand-drawn shapes (ids "a0", "a1", ...); imported ids are
# untouched. If you already deleted them in the view, this is a no-op.
drawn = [f["id"] for f in v.annotations["features"] if f["id"][0] == "a"]
for fid in reversed(drawn):
    v.delete_annotation(fid)
v.annotations

## 5. Clear

`clear_annotations()` removes everything — imported and drawn shapes.
M4 has no undo; the reset paths are `clear_annotations()` or a re-import.


In [ ]:
v.clear_annotations()
v.annotations